## Class Creation - Data Puller

In [1]:
# CLASS CREATION - STOCK_DATA_PULLER

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

class AlpacaDataPuller:
    """
    Pulls Stock Data from Alpaca API and establishes quartile-based baseline targets.
    """
    def __init__(
        self, 
        symbols: list[str], 
        api_key: str,
        secret_key: str,
        timeframe_amount: int = 5,
        timeframe_unit: str = "minute"
    ):
        self.symbols = symbols
        self.api_key = api_key
        self.secret_key = secret_key
        self.timeframe_amount = timeframe_amount
        self.timeframe_unit = timeframe_unit
        
        # Dictionary to store pulled self.dfs for different timeframes
        self.data = {}

        # Initialize Alpaca Client
        self.client = StockHistoricalDataClient(api_key, secret_key)

    def get_timeframe(self, amount=None, unit=None):
        if amount is None:
            amount = self.timeframe_amount
        if unit is None:
            unit = self.timeframe_unit

        unit = unit.lower()
        unit_map = {
            "minute": TimeFrameUnit.Minute,
            "min": TimeFrameUnit.Minute,
            "hour": TimeFrameUnit.Hour,
            "day": TimeFrameUnit.Day,
            "week": TimeFrameUnit.Week,
        }

        if unit not in unit_map:
            raise ValueError("timeframe_unit must be: minute, hour, day, or week")

        return TimeFrame(amount, unit_map[unit])

    def pull_symbol_data(self, symbol, start=None, end=None, timeframe_amount=None, timeframe_unit=None):
        if end is None:
            end_date = datetime.today()
        else:
            end_date = datetime(*end)

        if start is None:
            start_date = end_date - timedelta(days=365 * 10)
        else:
            start_date = datetime(*start)

        request = StockBarsRequest(
            symbol_or_symbols=symbol,
            timeframe=self.get_timeframe(timeframe_amount, timeframe_unit),
            start=start_date,
            end=end_date,
            feed=DataFeed.IEX  # Use IEX for free/paper subscriptions
        )

        bars = self.client.get_stock_bars(request)
        return bars.df

    def pull_symbols_data(self, symbols=None, start=None, end=None, timeframe_amount=None, timeframe_unit=None):
        if symbols is None:
            symbols = self.symbols

        data = {}
        for symbol in symbols:
            data[symbol] = self.pull_symbol_data(symbol, start, end, timeframe_amount, timeframe_unit)
        return data

    def pull_multi_timeframe_data(self, timeframes: dict, start=None, end=None):
        """
        Pulls data for multiple timeframes and stores it in self.data.
        timeframes dictionary format: {'daily': (1, 'day'), 'weekly': (1, 'week')}
        """
        for label, (amount, unit) in timeframes.items():
            print(f"Pulling data for timeframe: '{label}' ({amount} {unit})...")
            self.data[label] = self.pull_symbols_data(
                symbols=self.symbols,
                start=start,
                end=end,
                timeframe_amount=amount,
                timeframe_unit=unit
            )
        print("All timeframes pulled successfully!")

    # --- 5-CLASS QUARTILE BASELINE LOGIC ---

    def classify_return(self, r, q1, q3, thresh):
        if r < q1:
            return 0  # Strong Down
        elif r < -thresh:
            return 1  # Moderate Down
        elif r <= thresh:
            return 2  # Flat
        elif r <= q3:
            return 3  # Moderate Up
        else:
            return 4  # Strong Up

    def create_baselines(self, timeframe_label, zero_thresh=0.02, close_col='close'):
        """
        Establishes 5 classes based on quartiles and a zero threshold.
        Appends the 'Target' column to each symbol's DataFrame in self.data[timeframe_label].
        """
        if timeframe_label not in self.data:
            raise ValueError(f"Timeframe '{timeframe_label}' data has not been pulled yet.")
            
        print(f"\n=== 5-Class Baselines for Timeframe: {timeframe_label.upper()} ===")
        
        class_names = {
            0: "Strong Down (R < Q1)",
            1: "Moderate Down (Q1 <= R < -thresh)",
            2: "Flat (-thresh <= R <= thresh)",
            3: "Moderate Up (thresh < R <= Q3)",
            4: "Strong Up (R > Q3)"
        }
        
        for symbol, df in self.data[timeframe_label].items():
            df = df.copy()
            
            # Calculate next period's percentage return (t to t+1)
            # Note: We shift(-1) so that today's data points are mapped to tomorrow's actual return
            df['Next_Return'] = df[close_col].pct_change().shift(-1)
            
            # Clean returns for calculating statistics
            clean_returns = df['Next_Return'].dropna()
            
            # Compute Q1, Q3, and Flat Threshold
            q1 = clean_returns.quantile(0.25)
            q3 = clean_returns.quantile(0.75)
            avg_movement = clean_returns.abs().mean()
            thresh = zero_thresh * avg_movement
            
            print(f"\nSymbol: {symbol}")
            print(f"  Average Absolute Movement: {avg_movement * 100:.4f}%")
            print(f"  Flat Threshold (+/- {zero_thresh*100}% of Avg):  {thresh * 100:.4f}%")
            print(f"  Q1 (25th percentile):             {q1 * 100:.4f}%")
            print(f"  Q3 (75th percentile):             {q3 * 100:.4f}%")
            
            # Classify all returns
            df['Target'] = df['Next_Return'].apply(
                lambda x: self.classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            
            # Drop the last row which contains NaN target
            df = df.dropna(subset=['Target'])
            df['Target'] = df['Target'].astype(int)
            
            # Store the updated DataFrame back in self.data
            self.data[timeframe_label][symbol] = df
            
            # Print class distribution
            counts = df['Target'].value_counts().sort_index()
            total = len(df)
            for c, count in counts.items():
                print(f"  Class {c} | {class_names[c]:<35} : {count:<5} ({count/total*100:.2f}%)")


## SPY Example using AlpacaDataPuller

In [ ]:
## TEST CELL

# REQUIRED INPUTS
API_KEY = "PKEN63LDIFPD43FZN5JQXXXHXV"
SECRET_KEY = "3pmEQxuC9R5vGL4y6YSfDMWq9fgSM8nBZae8QGHyoE3C"
SYMBOLS = ['SPY']

# PULL SPY HISTORICALS INTO DICTIONARY BY TIME UNIT

puller_hrly = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="hour")

puller_daily = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="day")

puller_weekly = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="week")

SPY_dict = {
    'hourly': puller_hrly.pull_symbols_data(),
    'daily': puller_daily.pull_symbols_data(),
    'weekly': puller_weekly.pull_symbols_data()
}

In [2]:
# Pull data for 3 representative companies and Index (SPY,MU, TSLA, GOOG)

MULTI_SYMBOL = ["MU,GOOG,TSLA,SPY"]

API_KEY = "PKEN63LDIFPD43FZN5JQXXXHXV"
SECRET_KEY = "3pmEQxuC9R5vGL4y6YSfDMWq9fgSM8nBZae8QGHyoE3C"

# Initialize puller
puller = AlpacaDataPuller(MULTI_SYMBOL, API_KEY, SECRET_KEY)

# Define configurations to download (e.g. 1 hour, 1 day, 1 week bars)
timeframe_configs = {
    'hourly': (1, 'hour'),
    'daily': (1, 'day'),
    'weekly': (1, 'week')
}

# Pull 10 years of data (start dates will be limited by Alpaca for hourly)
puller.pull_multi_timeframe_data(timeframe_configs, start=(2016, 6, 20))

# Compute targets & prints metric summaries
puller.create_baselines('hourly')
puller.create_baselines('daily')
puller.create_baselines('weekly')


Pulling data for timeframe: 'hourly' (1 hour)...
Pulling data for timeframe: 'daily' (1 day)...
Pulling data for timeframe: 'weekly' (1 week)...
All timeframes pulled successfully!

=== 5-Class Baselines for Timeframe: HOURLY ===

Symbol: MU,GOOG,TSLA,SPY
  Average Absolute Movement: 0.5784%
  Flat Threshold (+/- 2.0% of Avg):  0.0116%
  Q1 (25th percentile):             -0.3036%
  Q3 (75th percentile):             0.3290%
  Class 0 | Strong Down (R < Q1)                : 11213 (25.00%)
  Class 1 | Moderate Down (Q1 <= R < -thresh)   : 9808  (21.87%)
  Class 2 | Flat (-thresh <= R <= thresh)       : 1169  (2.61%)
  Class 3 | Moderate Up (thresh < R <= Q3)      : 11450 (25.53%)
  Class 4 | Strong Up (R > Q3)                  : 11213 (25.00%)

=== 5-Class Baselines for Timeframe: DAILY ===

Symbol: MU,GOOG,TSLA,SPY
  Average Absolute Movement: 1.9519%
  Flat Threshold (+/- 2.0% of Avg):  0.0390%
  Q1 (25th percentile):             -1.0651%
  Q3 (75th percentile):             1.2520%
  Cl

In [3]:
import pandas as pd

'''create simple dataframe tables
symbol: Multiindex
'''
hourly_data = pd.DataFrame(puller.data['hourly']['MU,GOOG,TSLA,SPY'])
weekly_data = pd.DataFrame(puller.data['weekly']['MU,GOOG,TSLA,SPY'])
daily_data  = pd.DataFrame(puller.data['daily']['MU,GOOG,TSLA,SPY'])


In [4]:
hourly_data.reset_index(inplace=True)
weekly_data.reset_index(inplace=True)
daily_data.reset_index(inplace=True)

In [5]:
class feature_create:
    '''Class to help incorporate all methods for 
    metric creation'''

    def __init__(self,df):
        self.df = df
    
    def calculate_moving_avg(self):
        self.df['MU_50'] = self.df['close'].rolling(window=50).mean()
        self.df['MU_200'] = self.df['close'].rolling(window = 200).mean()
        self.df['ewm_50'] = self.df['close'].ewm(span=50,adjust = False).mean()
        self.df['ewm_200'] = self.df['close'].ewm(span=200,adjust = False).mean()

        return self.df

    def bollinger(self):
        self.df['BB_Mid'] = self.df['close'].rolling(window=20).mean()
        bb_std = self.df['close'].rolling(window=20).std()
        self.df['BB_Upper'] = self.df['BB_Mid'] + 2 * bb_std
        self.df['BB_Lower'] = self.df['BB_Mid'] - 2 * bb_std
        return self.df

# Calculate RSI and MACD
    def RSI(self):
        # 3. Momentum: Relative Strength Index (RSI - 14-day, Wilder's smoothing)
        delta = self.df['close'].diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
    
        # Wilder's smoothing uses alpha = 1 / period
        avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
        avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    
        rs = avg_gain / avg_loss
        self.df['RSI'] = 100 - (100 / (1 + rs))
        return self.df

    def MACD(self):
        ema_12 = self.df['close'].ewm(span=12, adjust=False).mean()
        ema_26 = self.df['close'].ewm(span=26, adjust=False).mean()
        self.df['MACD'] = ema_12 - ema_26
        self.df['MACD_Signal'] = self.df['MACD'].ewm(span=9, adjust=False).mean()
        self.df['MACD_Hist'] = self.df['MACD'] - self.df['MACD_Signal']
    
        return self.df


In [7]:
hourly_data_feature = feature_create(hourly_data)

# 2. Call the methods to calculate the features in-place
hourly_data_feature.calculate_moving_avg()
hourly_data_feature.bollinger()
hourly_data_feature.RSI()
hourly_data_feature.MACD()

# 3. View the updated DataFrame with the new columns!
hourly_data_feature.df


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,Next_Return,...,MU_200,ewm_50,ewm_200,BB_Mid,BB_Upper,BB_Lower,RSI,MACD,MACD_Signal,MACD_Hist
0,GOOG,2020-07-27 13:00:00+00:00,1517.695,1540.04,1515.950,1535.680,7684.0,151.0,1528.255299,-0.004369,...,NaN,1535.680000,1535.680000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
1,GOOG,2020-07-27 14:00:00+00:00,1536.785,1537.21,1528.245,1528.970,7231.0,120.0,1531.785692,-0.003630,...,NaN,1535.416863,1535.613234,NaN,NaN,NaN,NaN,-0.535271,-0.107054,-0.428217
2,GOOG,2020-07-27 15:00:00+00:00,1530.140,1530.14,1519.685,1523.420,5071.0,53.0,1524.665100,0.001904,...,NaN,1534.946398,1535.491908,NaN,NaN,NaN,NaN,-1.391277,-0.363899,-1.027379
3,GOOG,2020-07-27 16:00:00+00:00,1524.580,1528.05,1521.090,1526.320,3319.0,38.0,1523.898594,-0.001441,...,NaN,1534.608107,1535.400645,NaN,NaN,NaN,NaN,-1.814744,-0.654068,-1.160676
4,GOOG,2020-07-27 17:00:00+00:00,1526.325,1528.38,1524.120,1524.120,2707.0,31.0,1526.125600,0.002519,...,NaN,1534.196809,1535.288400,NaN,NaN,NaN,NaN,-2.301337,-0.983522,-1.317816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44848,TSLA,2026-06-18 15:00:00+00:00,388.400,391.49,387.455,390.710,153763.0,3201.0,389.426321,0.007474,...,413.759025,401.061589,407.839774,400.91400,412.994509,388.833491,36.168281,-3.079745,-1.323696,-1.756048
44849,TSLA,2026-06-18 16:00:00+00:00,391.050,393.95,390.580,393.630,146613.0,2779.0,392.802816,0.005132,...,413.601975,400.770154,407.698384,400.10300,411.830242,388.375758,41.374354,-3.091555,-1.677268,-1.414287
44850,TSLA,2026-06-18 17:00:00+00:00,393.630,397.15,392.640,395.650,161296.0,3141.0,395.459718,-0.002540,...,413.456325,400.569364,407.578499,399.55375,411.018916,388.088584,44.732480,-2.904436,-1.922702,-0.981735
44851,TSLA,2026-06-18 18:00:00+00:00,395.770,396.08,394.155,394.645,109208.0,1971.0,395.058678,0.014811,...,413.287500,400.337036,407.449808,398.97775,410.199339,387.756161,43.400483,-2.804906,-2.099142,-0.705763
